# Supplementary Figures 1-3 — behavioural measures

**S1** Number of fixations needed to reach the face.
**S2** Detection latency on trials where the first fixation already landed on the
face, face detection task only.
**S3** Scan path ratio, moving-window task only.

Which measures are log-transformed is a deliberate choice. Latency and scan path
ratio are ratios or durations: bounded below by zero, unbounded above, and
multiplicative, so they are tested on logs. Fixation counts and percentages are
bounded and not multiplicative, so they are tested raw. Raw means are always
reported for description.

In [ ]:
import sys
from pathlib import Path

# The notebooks live next to the code, so the package is importable without
# installing anything. Nothing outside this release is referenced.
sys.path.insert(0, str(Path.cwd().parent / "code"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from facescenes import config as cfg
from facescenes import data as D
from facescenes import behaviour as B
from facescenes import encoding as E
from facescenes import onset_gaze as G
from facescenes import plotting as P
from facescenes import stats as S

P.use_style()
print("stimuli    :", len(D.stimulus_names()))
for t in cfg.TASKS:
    print(f"{t:14s}: {len(D.participant_ids(t))} participants, "
          f"{len(D.analysable_images(t))} images")

In [ ]:
def paired_panel(task, column, ylabel, fname, result, step=None, df=None,
                 resume=None):
    """One panel: per-participant means in the two conditions.

    `resume` breaks the y-axis, as in Figure 1b: a stub carrying the zero tick,
    the break, then the data range from `resume` up. Used where the effect is a
    few ms on a mean of ~190 ms and would otherwise be invisible.
    """
    if df is None:
        df = B.trial_table(task)
    a, b = B.participant_means(df, column)

    top = np.nanmax([a, b])
    if step is None:
        step = 10 ** np.floor(np.log10(top)) / 2
    # Headroom for the significance marker, except on a broken axis, where the
    # range above `resume` is already tall relative to the data.
    hi = np.ceil((top if resume else top * 1.12) / step) * step
    lo = 0 if resume is None else resume

    if resume is None:
        fig, ax = plt.subplots(figsize=(5, 7))
        stub = None
    else:
        fig, (ax, stub) = plt.subplots(2, 1, sharex=True, figsize=(5, 7),
                                       gridspec_kw=dict(height_ratios=[18, 1],
                                                        hspace=.06))

    for a_ in ([ax] if stub is None else [ax, stub]):
        for i in range(len(a)):
            a_.plot([0, 1], [a[i], b[i]], color="grey", lw=.6, alpha=.5, zorder=1)
        box = a_.boxplot([a, b], positions=[0, 1], widths=.5, patch_artist=True,
                         showfliers=False, zorder=2,
                         medianprops=dict(color="k", lw=2))
        for patch, cond in zip(box["boxes"], cfg.CONDITIONS):
            patch.set_facecolor(P.condition_colours()[cond]); patch.set_edgecolor("k")
        P.clean(a_)

    ax.set_ylim(lo, hi)
    ax.set_yticks(np.arange(lo, hi + step / 2, step))
    ax.set_xticks([0, 1])
    P.title(ax, cfg.TASK_LABEL[task], fontsize=P.FONT - 6, pad=12)
    ax.text(.5, .96, P.significance(result["p"]), ha="center", va="top",
            transform=ax.transAxes, fontsize=P.FONT - 6)
    ax.set_ylabel(ylabel, fontsize=P.FONT - 8)
    labels = [cfg.COND_LABEL[c] for c in cfg.CONDITIONS]

    if stub is None:
        ax.set_xticklabels(labels, fontsize=P.FONT - 8)
    else:
        stub.set_ylim(0, lo * .04)          # a sliver of the axis, so 0 is shown
        stub.set_yticks([0])
        stub.set_xticklabels(labels, fontsize=P.FONT - 8)
        ax.spines["bottom"].set_visible(False)
        stub.spines["top"].set_visible(False)
        ax.tick_params(axis="x", bottom=False)
        P.break_marker(ax, stub)
    src = B.participant_table(df, column).rename(
        columns=lambda c: f"{c}_mean_{column}").reset_index()
    P.save(fig, fname, src); plt.show()

    r = result
    print(f"{cfg.TASK_LABEL[task]} — {ylabel}")
    print(f"  raw means  {r['mean_a']:.3f} vs {r['mean_b']:.3f}")
    if "geo_a" in r:
        print(f"  geometric  {r['geo_a']:.3f} vs {r['geo_b']:.3f} "
              f"({r['pct_change']:+.2f} %)")
    print(f"  t({r['n']-1}) = {r['t']:.3f}, p = {r['p']:.4g}, dz = {r['dz']:.2f}, "
          f"95% CI [{r['dz_ci'][0]:.2f}, {r['dz_ci'][1]:.2f}]")


for task, fname in (("facedetection", "suppfig1a_fixation_counts_face_detection.png"),
                    ("movingwindow", "suppfig1b_fixation_counts_moving_window.png")):
    paired_panel(task, "n_fixations", "Fixations to reach the face", fname,
                 B.fixation_count(task), step=1 if task == "facedetection" else 2)

## S2 — detection latency on trials reached in a single fixation

Trials where the first fixation beginning more than 100 ms after scene onset
already landed on the face, so the search ended on one saccade. What is plotted
is the LATENCY of that fixation, not how often it happened; the hit rate is
printed alongside. Face detection task only: in the moving-window task only 5.9 %
of trials end this way, and ten participants have no such trial in one of the two
conditions, so the comparison rests on 28 of 38 participants.

Display is on the raw scale, the test on logs, as for detection latency in
Figure 1.

In [ ]:
task = "facedetection"
ff = B.first_fixation_on_face(task)
print(f"{cfg.TASK_LABEL[task]}: {ff['n_hit']}/{ff['n_total']} trials "
      f"({ff['pct_overall']:.1f} %) ended on the first fixation")

df = B.trial_table(task)
hits = df[df["n_fixations"] == 1]
paired_panel(task, "latency_ms",
             "Detection latency with a\nsingle fixation (ms)",
             "suppfig2_first_fixation_latency_face_detection.png",
             ff["latency"], step=40, resume=160, df=hits)

## S3 — scan path ratio

Total distance travelled between successive fixations, divided by the
straight-line distance from the centre to the face. A value of 1 means the
observer went straight there. Reported for the moving-window task only: in the
face detection task observers reach the face in roughly one fixation, so there is
barely a path to measure.

In [ ]:
paired_panel("movingwindow", "scan_path_ratio", "Scan path ratio",
             "suppfig3_scanpath_ratio_moving_window.png",
             B.scan_path_ratio("movingwindow"), step=1)